# Feature knowledge distillation

We compare supervised learning and feature distillation from RT-DETRv2, starting both ResNet18 students from the same weights. `RUN_TRAINING = False` reads the existing results.

In [ ]:
from pathlib import Path
import sys
import json
import random
import numpy as np
import torch
from torch.utils.data import DataLoader

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path[:0] = [str(REPO_ROOT), str(REPO_ROOT / 'students/anna_student_resnet18/src')]

from src.utils.checkpoint import load_checkpoint
from src.engine.inference import prepare_validation_files
from sixray_student.data import SixRayStudentDataset, detection_collate_fn, load_split
from sixray_student.model import build_student_model
from sixray_student.config import CLASS_NAMES, BBOX_ENCODING
from sixray_student.metrics import evaluate_map
from sixray_student.train_utils import train_one_epoch, make_grad_scaler

## Paths and settings

In [ ]:
ARTIFACT_ROOT = REPO_ROOT / "artifacts"
PROJECT_DIR = ARTIFACT_ROOT / "DatasetAPAI/SIXray_Project"
DATA_DIR = ARTIFACT_ROOT / "DatasetAPAI/subset_clean"

SPLIT_PATH = PROJECT_DIR / "splits/split_seed42_train10500_val300pos1200neg.json"
TEACHER_PATH = PROJECT_DIR / "checkpoints_angelo/02_rtdetr_teacher_oversampling_angelo_best"
INITIAL_PATH = ARTIFACT_ROOT / "experiments/feature_kd_20261016T214646677055Z/initial_student.pth"

RUN_TRAINING = False
RESULTS_DIR = ARTIFACT_ROOT / "experiments/matched_20261016T221317947683Z"

if RUN_TRAINING:
    from datetime import datetime
    RESULTS_DIR = ARTIFACT_ROOT / "experiments" / datetime.now().strftime("matched_%Y%m%d_%H%M%S")
    RESULTS_DIR.mkdir(parents=True, exist_ok=False)

SEED = 42
ADAPTER_SEED = 43
EPOCHS = 50
BATCH_SIZE = 2
NUM_WORKERS = 2
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MIN_LR = 1e-6
KD_WEIGHT = 1.0
BACKGROUND_WEIGHT = 0.1

SCORE_THRESHOLD = 0.001
NMS_IOU = 0.5
MAX_DETECTIONS = 100

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Split and loaders

In [ ]:
train_indices, val_indices, _ = load_split(SPLIT_PATH)

if RUN_TRAINING:
    prepare_validation_files(
        ARTIFACT_ROOT / "DatasetAPAI/subset_clean.zip",
        DATA_DIR,
        train_indices + val_indices
    )

    train_data = SixRayStudentDataset(
        DATA_DIR / "train/images", DATA_DIR / "train.json", train_indices
    )
    val_data = SixRayStudentDataset(
        DATA_DIR / "train/images", DATA_DIR / "train.json", val_indices
    )

    train_loader = DataLoader(
        train_data, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, collate_fn=detection_collate_fn,
        generator=torch.Generator().manual_seed(SEED)
    )
    val_loader = DataLoader(
        val_data, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, collate_fn=detection_collate_fn,
        generator=torch.Generator().manual_seed(SEED)
    )

## Teacher features

We freeze the oversampling teacher backbone. A training-only 1 × 1 convolution maps student features from 512 to 2048 channels on the shared 20 × 20 grid.

We compare channel-normalized features. To define foreground cells, we project boxes onto the grid, floor the left/top boundaries, ceil the right/bottom boundaries, and take the union of the boxes. We average foreground and background distances separately for each image. Negative images contribute background only.

KD loss = foreground loss + 0.1 × background loss. Total loss = objectness + 10 × box loss + classification + KD loss. The teacher receives resized [0, 1] images; the student applies ImageNet normalization internally.

In [ ]:
if RUN_TRAINING:
    _, teacher, _ = load_checkpoint(TEACHER_PATH, device=DEVICE, strict=True)
    teacher.requires_grad_(False)
    teacher.eval()

## Matched training

We use the same initial weights and shuffle seed for both students, initialize the adapter with a separate seed, and select checkpoints by validation mAP.

In [ ]:
def train_student(name, use_kd=False):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    student = build_student_model(pretrained_backbone=False).to(DEVICE)
    student.load_state_dict(
        torch.load(INITIAL_PATH, map_location=DEVICE, weights_only=True)
    )

    # Only the KD student needs an adapter
    adapter = None
    parameters = list(student.parameters())

    if use_kd:
        # Keep adapter initialization separate from the training seed
        with torch.random.fork_rng():
            torch.manual_seed(ADAPTER_SEED)
            adapter = torch.nn.Conv2d(512, 2048, 1, bias=False).to(DEVICE)
        parameters += list(adapter.parameters())

    train_loader.generator.manual_seed(SEED)
    val_loader.generator.manual_seed(SEED)

    optimizer = torch.optim.AdamW(parameters, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LR)
    scaler = make_grad_scaler(DEVICE)

    run_dir = RESULTS_DIR / name
    run_dir.mkdir()
    history = []
    best_map = -1

    for epoch in range(1, EPOCHS + 1):
        train_metrics = train_one_epoch(
            student, train_loader, optimizer, DEVICE, epoch, scaler,
            use_amp=True, grad_clip_norm=1.0, log_every_n_batches=0,
            scheduler=scheduler,
            teacher=teacher if use_kd else None,
            adapter=adapter,
            kd_weight=KD_WEIGHT,
            background_weight=BACKGROUND_WEIGHT
        )

        validation = evaluate_map(
            student, val_loader, DEVICE,
            confidence_threshold=SCORE_THRESHOLD,
            nms_iou_threshold=NMS_IOU,
            max_detections=MAX_DETECTIONS,
            use_amp=False,
            original_coordinates=True
        )

        history.append({
            "epoch": epoch,
            "train": train_metrics,
            "validation": validation
        })

        # Save the best model and its validation metrics.
        if validation["map"] > best_map:
            best_map = validation["map"]
            checkpoint = {
                "epoch": epoch,
                "model_state_dict": student.state_dict(),
                "adapter_state_dict": adapter.state_dict() if use_kd else None,
                "validation": validation,
                "class_names": CLASS_NAMES,
                "image_size": 640,
                "grid_size": 20,
                "num_classes": 5,
                "num_boxes": 2,
                "bbox_encoding": BBOX_ENCODING,
                "best_checkpoint_metric": "map"
            }
            torch.save(checkpoint, run_dir / "student_best_map.pth")

        with open(run_dir / "history.json", "w") as f:
            json.dump(history, f, indent=2)

        print(
            f"{name} {epoch}/{EPOCHS}: "
            f"detection={train_metrics['detection_loss']:.4f}, "
            f"KD={train_metrics['kd_loss']:.4f}, mAP={validation['map']:.4f}"
        )

## Supervised student

In [ ]:
if RUN_TRAINING:
    with open(RESULTS_DIR / 'settings.json', 'x') as output:
        json.dump({
            'seed': SEED, 'adapter_seed': ADAPTER_SEED, 'split_path': str(SPLIT_PATH),
            'data_dir': str(DATA_DIR), 'teacher_path': str(TEACHER_PATH), 'initial_path': str(INITIAL_PATH),
            'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'num_workers': NUM_WORKERS,
            'optimizer': 'AdamW', 'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY,
            'scheduler': 'CosineAnnealingLR (T_max=50 epochs parameter; stepped inside train_one_epoch)', 'min_lr': MIN_LR, 'grad_clip_norm': 1.0,
            'kd_weight': KD_WEIGHT, 'background_weight': BACKGROUND_WEIGHT,
            'preprocessing': 'resize_only_640; student internal ImageNet normalization',
            'training_amp': DEVICE.type == 'cuda', 'evaluation_amp': False,
            'score_threshold': SCORE_THRESHOLD, 'nms_iou': NMS_IOU, 'max_detections': MAX_DETECTIONS,
            'evaluation_frame': 'original_image_xyxy', 'selection_metric': 'map',
            'control_checkpoint': 'control/student_best_map.pth', 'kd_checkpoint': 'kd/student_best_map.pth',
        }, output, indent=2)
    train_student('control', use_kd=False)

## Student with distillation

In [ ]:
if RUN_TRAINING:
    train_student('kd', use_kd=True)

## Results

In [6]:
print(f"{'Model':<12} {'Epoch':>6} {'mAP':>8} {'AP50':>8} {'AP75':>8}")
for name in ('control', 'kd'):
    checkpoint = torch.load(RESULTS_DIR / name / 'student_best_map.pth', map_location='cpu', weights_only=False)
    metrics = checkpoint['validation']
    print(f"{name:<12} {checkpoint['epoch']:>6} {metrics['map']:>8.4f} "
          f"{metrics['map_50']:>8.4f} {metrics['map_75']:>8.4f}")

Model         Epoch      mAP     AP50     AP75
control          46   0.3813   0.6655   0.3859
kd               45   0.4043   0.6882   0.4075
